<a href="https://colab.research.google.com/github/9569791913harsh-hub/Event2Career/blob/main/Event2Career.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install flask qrcode[pil] pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import os, sqlite3, secrets, hashlib, hmac, io, csv
from datetime import datetime
from functools import wraps
from flask import Flask, request, redirect, url_for, session, flash, render_template_string, send_file, abort
import qrcode

APP = Flask(__name__)
APP.secret_key = os.environ.get("EVENT2CAREER_SECRET", secrets.token_hex(32))
DB = "event2career.db"
ROLES = ("Student", "Organizer", "Faculty", "Admin")

def db():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA foreign_keys=ON")
    return con

def hash_pw(pw, salt=None):
    salt = salt or secrets.token_hex(16)
    digest = hashlib.pbkdf2_hmac("sha256", pw.encode(), salt.encode(), 180000).hex()
    return salt + "$" + digest

def verify_pw(pw, stored):
    try:
        salt, digest = stored.split("$", 1)
        test = hashlib.pbkdf2_hmac("sha256", pw.encode(), salt.encode(), 180000).hex()
        return hmac.compare_digest(test, digest)
    except Exception:
        return False

def init_db():
    con = db()
    con.executescript("""
    CREATE TABLE IF NOT EXISTS users(
      id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, email TEXT UNIQUE NOT NULL,
      password_hash TEXT NOT NULL, role TEXT NOT NULL, department TEXT DEFAULT 'Data Science',
      interests TEXT DEFAULT 'Technology, AI', timetable TEXT DEFAULT '');
    CREATE TABLE IF NOT EXISTS events(
      id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT NOT NULL, description TEXT NOT NULL,
      category TEXT NOT NULL, event_date TEXT NOT NULL, start_time TEXT NOT NULL, end_time TEXT NOT NULL,
      venue TEXT NOT NULL, capacity INTEGER NOT NULL, deadline TEXT NOT NULL,
      eligibility TEXT DEFAULT 'All students', organizer_id INTEGER NOT NULL,
      status TEXT NOT NULL DEFAULT 'Pending', created_at TEXT NOT NULL,
      FOREIGN KEY(organizer_id) REFERENCES users(id));
    CREATE TABLE IF NOT EXISTS registrations(
      id INTEGER PRIMARY KEY AUTOINCREMENT, event_id INTEGER NOT NULL, student_id INTEGER NOT NULL,
      qr_token TEXT UNIQUE NOT NULL, registered_at TEXT NOT NULL,
      UNIQUE(event_id, student_id), FOREIGN KEY(event_id) REFERENCES events(id),
      FOREIGN KEY(student_id) REFERENCES users(id));
    CREATE TABLE IF NOT EXISTS attendance(
      id INTEGER PRIMARY KEY AUTOINCREMENT, registration_id INTEGER UNIQUE NOT NULL,
      scanned_at TEXT NOT NULL, FOREIGN KEY(registration_id) REFERENCES registrations(id));
    CREATE TABLE IF NOT EXISTS certificates(
      id INTEGER PRIMARY KEY AUTOINCREMENT, registration_id INTEGER UNIQUE NOT NULL,
      certificate_no TEXT UNIQUE NOT NULL, issued_at TEXT NOT NULL,
      FOREIGN KEY(registration_id) REFERENCES registrations(id));
    CREATE TABLE IF NOT EXISTS feedback(
      id INTEGER PRIMARY KEY AUTOINCREMENT, event_id INTEGER NOT NULL, student_id INTEGER NOT NULL,
      rating INTEGER NOT NULL CHECK(rating BETWEEN 1 AND 5), comments TEXT DEFAULT '',
      submitted_at TEXT NOT NULL, UNIQUE(event_id, student_id),
      FOREIGN KEY(event_id) REFERENCES events(id), FOREIGN KEY(student_id) REFERENCES users(id));
    """)
    if con.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0:
        demo = [
          ("Demo Admin","admin@event2career.demo","Admin123!","Admin","Administration","All"),
          ("Demo Faculty","faculty@event2career.demo","Faculty123!","Faculty","Data Science","AI, Technology"),
          ("Demo Organizer","organizer@event2career.demo","Organizer123!","Organizer","Data Science","AI, Coding"),
          ("Demo Student","student@event2career.demo","Student123!","Student","Data Science","AI, Coding, Career")
        ]
        for name,email,pw,role,dept,interest in demo:
            con.execute("INSERT INTO users(name,email,password_hash,role,department,interests) VALUES(?,?,?,?,?,?)",
                        (name,email,hash_pw(pw),role,dept,interest))
        org = con.execute("SELECT id FROM users WHERE role='Organizer'").fetchone()["id"]
        samples = [
          ("AI & Data Science Hackathon","Build a useful prototype with a team. Bring your laptop and ideas.","Hackathon","2026-11-15","10:00","16:00","Innovation Lab",100,"2026-11-14","All students",org,"Approved"),
          ("Career Readiness Workshop","Resume, interview and professional communication practice.","Workshop","2026-11-20","11:00","13:00","Seminar Hall A",80,"2026-11-19","All students",org,"Approved"),
          ("Cloud Computing Seminar","Explore cloud concepts, deployment and industry use cases.","Seminar","2026-11-25","14:00","16:00","Room B-204",70,"2026-11-24","All students",org,"Approved")
        ]
        for e in samples:
            con.execute("""INSERT INTO events(title,description,category,event_date,start_time,end_time,venue,capacity,deadline,eligibility,organizer_id,status,created_at)
                           VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)""", (*e,datetime.now().isoformat(timespec="seconds")))
    con.commit()
    con.close()

def current_user():
    if "user_id" not in session: return None
    con=db(); u=con.execute("SELECT * FROM users WHERE id=?",(session["user_id"],)).fetchone(); con.close()
    return u

def login_required(fn):
    @wraps(fn)
    def wrap(*args, **kwargs):
        if not current_user(): return redirect(url_for("login"))
        return fn(*args, **kwargs)
    return wrap

def roles_required(*roles):
    def deco(fn):
        @wraps(fn)
        def wrap(*args, **kwargs):
            u=current_user()
            if not u: return redirect(url_for("login"))
            if u["role"] not in roles: abort(403)
            return fn(*args, **kwargs)
        return wrap
    return deco

BASE = """
<!doctype html><html><head><meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ title }} | Event2Career</title>
<style>
:root{--navy:#14213d;--blue:#2563eb;--light:#f4f7fb;--ink:#182230;--muted:#64748b;--line:#e2e8f0}
*{box-sizing:border-box}body{margin:0;font-family:Inter,Arial,sans-serif;background:var(--light);color:var(--ink)}
header{background:linear-gradient(115deg,#14213d,#263e70);color:white;padding:18px max(5%,calc((100% - 1180px)/2));display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap}
.brand{font-size:22px;font-weight:800;letter-spacing:.2px}.brand span{color:#93c5fd}
nav{display:flex;gap:10px;flex-wrap:wrap}nav a{color:white;text-decoration:none;font-size:13px;padding:8px 10px;border-radius:9px}nav a:hover{background:#ffffff22}
main{max-width:1180px;margin:26px auto;padding:0 18px}.hero{padding:28px;border-radius:18px;color:white;background:linear-gradient(120deg,#1d4ed8,#6d28d9);margin-bottom:22px}
.hero h1{font-size:clamp(26px,4vw,42px);margin:0 0 8px}.hero p{margin:0;color:#e0e7ff;max-width:720px;line-height:1.6}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(245px,1fr));gap:16px}.card{background:white;border:1px solid var(--line);border-radius:15px;padding:19px;box-shadow:0 5px 20px #14213d08}
.card h3{margin:8px 0}.muted{color:var(--muted);font-size:13px;line-height:1.55}.pill{display:inline-block;background:#dbeafe;color:#1d4ed8;border-radius:99px;padding:5px 10px;font-size:12px;font-weight:700}
.btn,button{display:inline-block;border:0;border-radius:10px;background:var(--blue);color:white;padding:10px 14px;font-weight:700;text-decoration:none;cursor:pointer;font-size:14px}.btn.secondary{background:#e2e8f0;color:#1e293b}.btn.green{background:#15803d}.btn.red{background:#b91c1c}
form{display:grid;gap:12px}label{font-weight:700;font-size:13px}input,select,textarea{width:100%;padding:11px;border:1px solid #cbd5e1;border-radius:9px;margin-top:5px;font:inherit;background:white}textarea{min-height:95px}
.row{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px}.actions{display:flex;gap:8px;flex-wrap:wrap;margin-top:14px}
table{width:100%;border-collapse:collapse;background:white;border-radius:12px;overflow:hidden}th,td{text-align:left;padding:12px;border-bottom:1px solid var(--line);font-size:13px}th{background:#eff6ff}
.alert{padding:12px 15px;border-radius:10px;background:#fef3c7;margin:12px 0}.stat{font-size:28px;font-weight:800;color:#1d4ed8}.section-title{margin:26px 0 13px}footer{text-align:center;padding:25px;color:#64748b;font-size:12px}
@media(max-width:600px){header{padding:16px}main{margin:16px auto}.hero{padding:22px}td,th{padding:8px}}
</style></head><body>
<header><div class="brand">Event2<span>Career</span></div><nav>
<a href="{{ url_for('index') }}">Events</a>
{% if user %}<a href="{{ url_for('dashboard') }}">Dashboard</a>
{% if user.role in ['Organizer','Admin'] %}<a href="{{ url_for('create_event') }}">Create Event</a>{% endif %}
{% if user.role in ['Faculty','Admin'] %}<a href="{{ url_for('approvals') }}">Approvals</a>{% endif %}
{% if user.role=='Admin' %}<a href="{{ url_for('admin_users') }}">Users</a>{% endif %}
<a href="{{ url_for('logout') }}">Logout</a>{% else %}<a href="{{ url_for('login') }}">Login</a><a href="{{ url_for('register_account') }}">Create account</a>{% endif %}
</nav></header><main>
{% with messages=get_flashed_messages() %}{% for m in messages %}<div class="alert">{{m}}</div>{% endfor %}{% endwith %}
{{ body|safe }}
</main><footer>Event2Career · College Event Management System · Academic prototype</footer></body></html>
"""

def page(title, body, **kwargs):
    return render_template_string(BASE, title=title, body=render_template_string(body, **kwargs), user=current_user())

def conflict_messages(con, date, start, end, venue, ignore_id=None):
    q="SELECT * FROM events WHERE event_date=? AND status IN ('Pending','Approved')"
    args=[date]
    if ignore_id:
        q += " AND id<>?"; args.append(ignore_id)
    rows=con.execute(q,args).fetchall()
    warnings=[]
    for e in rows:
        overlap = start < e["end_time"] and e["start_time"] < end
        if overlap and e["venue"].strip().lower()==venue.strip().lower():
            warnings.append(f"Venue conflict with '{e['title']}' ({e['start_time']}-{e['end_time']}, {e['venue']}).")
        elif overlap:
            warnings.append(f"Schedule overlap with '{e['title']}' ({e['start_time']}-{e['end_time']}).")
    return warnings

@APP.route("/")
def index():
    u=current_user()
    q=request.args.get("q","").strip()
    cat=request.args.get("category","")
    con=db()
    sql="SELECT e.*,u.name AS organizer FROM events e JOIN users u ON u.id=e.organizer_id WHERE e.status='Approved'"
    args=[]
    if q:
        sql+=" AND (e.title LIKE ? OR e.description LIKE ? OR e.venue LIKE ?)"; args += [f"%{q}%"]*3
    if cat:
        sql+=" AND e.category=?"; args.append(cat)
    sql+=" ORDER BY e.event_date,e.start_time"
    events=con.execute(sql,args).fetchall()
    categories=[r["category"] for r in con.execute("SELECT DISTINCT category FROM events WHERE status='Approved' ORDER BY category")]
    con.close()
    body="""<section class="hero"><h1>Your campus. Your opportunities.</h1><p>Discover workshops, hackathons, seminars and campus activities. Register online, keep your QR pass handy and build your learning journey with Event2Career.</p><div class="actions"><a class="btn secondary" href="{{url_for('dashboard')}}">Open my dashboard</a></div></section>
    <form method="get" class="card"><div class="row"><div><label>Search events</label><input name="q" value="{{q}}" placeholder="Try AI, career, venue..."></div><div><label>Category</label><select name="category"><option value="">All categories</option>{% for c in categories %}<option {% if c==cat %}selected{% endif %}>{{c}}</option>{% endfor %}</select></div></div><button>Search events</button></form>
    <h2 class="section-title">Upcoming approved events <span class="muted">({{events|length}})</span></h2>
    <div class="grid">{% for e in events %}<article class="card"><span class="pill">{{e.category}}</span><h3>{{e.title}}</h3><p class="muted">{{e.description}}</p><p>📅 {{e.event_date}} · {{e.start_time}}–{{e.end_time}}</p><p>📍 {{e.venue}}</p><p class="muted">Organizer: {{e.organizer}} · Capacity: {{e.capacity}}</p><a class="btn" href="{{url_for('event_detail',event_id=e.id)}}">View event</a></article>{% else %}<div class="card">No events match your search.</div>{% endfor %}</div>"""
    return page("Discover Events",body,events=events,categories=categories,q=q,cat=cat)

@APP.route("/register", methods=["GET","POST"])
def register_account():
    if request.method=="POST":
        name=request.form.get("name","").strip(); email=request.form.get("email","").strip().lower()
        pw=request.form.get("password",""); interests=request.form.get("interests","Technology, AI")
        if not name or "@" not in email or len(pw)<8:
            flash("Enter a name, valid email, and password of at least 8 characters.")
        else:
            con=db()
            try:
                con.execute("INSERT INTO users(name,email,password_hash,role,interests) VALUES(?,?,?,?,?)",(name,email,hash_pw(pw),"Student",interests))
                con.commit(); flash("Account created. Please log in."); con.close(); return redirect(url_for("login"))
            except sqlite3.IntegrityError:
                con.close(); flash("That email is already registered.")
    body="""<div class="card" style="max-width:600px;margin:auto"><h2>Create student account</h2><p class="muted">New self-registered accounts use the Student role. Demo roles are available in the notebook instructions.</p><form method="post"><label>Full name<input required name="name"></label><label>Email<input required type="email" name="email"></label><label>Password (minimum 8 characters)<input required type="password" minlength="8" name="password"></label><label>Interests (comma-separated)<input name="interests" value="Technology, AI, Career"></label><button>Create account</button></form></div>"""
    return page("Create Account",body)

@APP.route("/login", methods=["GET","POST"])
def login():
    if request.method=="POST":
        email=request.form.get("email","").strip().lower(); pw=request.form.get("password","")
        con=db(); u=con.execute("SELECT * FROM users WHERE email=?",(email,)).fetchone(); con.close()
        if u and verify_pw(pw,u["password_hash"]):
            session.clear(); session["user_id"]=u["id"]; flash("Welcome, "+u["name"]+"!"); return redirect(url_for("dashboard"))
        flash("Invalid email or password.")
    body="""<div class="card" style="max-width:560px;margin:auto"><h2>Sign in</h2><form method="post"><label>Email<input type="email" name="email" required></label><label>Password<input type="password" name="password" required></label><button>Login</button></form><h3>Demo accounts</h3><p class="muted">Admin: admin@event2career.demo / Admin123!<br>Faculty: faculty@event2career.demo / Faculty123!<br>Organizer: organizer@event2career.demo / Organizer123!<br>Student: student@event2career.demo / Student123!</p></div>"""
    return page("Login",body)

@APP.route("/logout")
def logout():
    session.clear(); flash("You have been logged out."); return redirect(url_for("index"))

@APP.route("/dashboard")
@login_required
def dashboard():
    u=current_user(); con=db()
    if u["role"]=="Student":
        regs=con.execute("""SELECT r.id AS rid,r.qr_token,e.*,a.scanned_at,c.certificate_no
        FROM registrations r JOIN events e ON e.id=r.event_id LEFT JOIN attendance a ON a.registration_id=r.id
        LEFT JOIN certificates c ON c.registration_id=r.id WHERE r.student_id=? ORDER BY e.event_date""",(u["id"],)).fetchall()
        interest=[x.strip().lower() for x in (u["interests"] or "").split(",") if x.strip()]
        recs=con.execute("SELECT * FROM events WHERE status='Approved' AND event_date>=date('now') ORDER BY event_date").fetchall()
        recs=sorted(recs,key=lambda e:sum(1 for x in interest if x in (e["category"]+" "+e["title"]+" "+e["description"]).lower()),reverse=True)[:4]
        feedback_events=con.execute("""SELECT e.* FROM events e JOIN registrations r ON r.event_id=e.id
          LEFT JOIN feedback f ON f.event_id=e.id AND f.student_id=r.student_id
          WHERE r.student_id=? AND f.id IS NULL""",(u["id"],)).fetchall()
        con.close()
        body="""<section class="hero"><h1>Student dashboard</h1><p>Welcome, {{u.name}}. Explore recommended events, manage your registrations and access eligible certificates.</p></section>
        <div class="grid"><div class="card"><div class="stat">{{regs|length}}</div><div class="muted">Your registrations</div></div><div class="card"><div class="stat">{{regs|selectattr('scanned_at')|list|length}}</div><div class="muted">Attendance records</div></div><div class="card"><div class="stat">{{regs|selectattr('certificate_no')|list|length}}</div><div class="muted">Certificates</div></div></div>
        <h2 class="section-title">Recommended for you</h2><div class="grid">{% for e in recs %}<div class="card"><span class="pill">{{e.category}}</span><h3>{{e.title}}</h3><p class="muted">{{e.event_date}} · {{e.venue}}</p><a class="btn" href="{{url_for('event_detail',event_id=e.id)}}">Explore</a></div>{% else %}<div class="card">No recommendations yet.</div>{% endfor %}</div>
        <h2 class="section-title">My registrations</h2><div class="card">{% if regs %}<table><tr><th>Event</th><th>Date</th><th>QR pass</th><th>Attendance</th><th>Certificate</th></tr>{% for r in regs %}<tr><td>{{r.title}}</td><td>{{r.event_date}}</td><td><a href="{{url_for('qr_pass',registration_id=r.rid)}}">View QR</a></td><td>{{'Attended' if r.scanned_at else 'Not marked'}}</td><td>{% if r.certificate_no %}<a href="{{url_for('certificate',registration_id=r.rid)}}">Download</a>{% else %}Not issued{% endif %}</td></tr>{% endfor %}</table>{% else %}<p class="muted">You have not registered for any events yet.</p>{% endif %}</div>
        <h2 class="section-title">Submit feedback</h2><div class="grid">{% for e in feedback_events %}<div class="card"><h3>{{e.title}}</h3><form method="post" action="{{url_for('feedback',event_id=e.id)}}"><label>Rating (1–5)<select name="rating">{% for n in range(5,0,-1) %}<option>{{n}}</option>{% endfor %}</select></label><label>Comments<textarea name="comments"></textarea></label><button>Submit feedback</button></form></div>{% endfor %}</div>"""
        return page("Student Dashboard",body,u=u,regs=regs,recs=recs,feedback_events=feedback_events)
    if u["role"]=="Organizer":
        events=con.execute("SELECT * FROM events WHERE organizer_id=? ORDER BY created_at DESC",(u["id"],)).fetchall()
        body="""<section class="hero"><h1>Organizer dashboard</h1><p>Create event proposals, check potential clashes and monitor registrations.</p><a class="btn secondary" href="{{url_for('create_event')}}">+ Create event</a></section><div class="grid">{% for e in events %}<div class="card"><span class="pill">{{e.status}}</span><h3>{{e.title}}</h3><p class="muted">{{e.event_date}} · {{e.start_time}}–{{e.end_time}} · {{e.venue}}</p><p>{{e.description}}</p><a class="btn" href="{{url_for('event_detail',event_id=e.id)}}">Open</a> <a class="btn secondary" href="{{url_for('event_report',event_id=e.id)}}">Analytics</a></div>{% else %}<div class="card">No events created yet.</div>{% endfor %}</div>"""
        con.close(); return page("Organizer Dashboard",body,events=events)
    if u["role"]=="Faculty":
        pending=con.execute("SELECT e.*,u.name organizer FROM events e JOIN users u ON u.id=e.organizer_id WHERE e.status='Pending' ORDER BY e.created_at").fetchall()
        approved=con.execute("SELECT COUNT(*) n FROM events WHERE status='Approved'").fetchone()["n"]
        con.close()
        body="""<section class="hero"><h1>Faculty dashboard</h1><p>Review event proposals and help ensure event schedules and venues are coordinated.</p></section><div class="grid"><div class="card"><div class="stat">{{pending|length}}</div><div class="muted">Pending proposals</div></div><div class="card"><div class="stat">{{approved}}</div><div class="muted">Approved events</div></div></div><h2 class="section-title">Proposals awaiting review</h2><div class="grid">{% for e in pending %}<div class="card"><span class="pill">{{e.category}}</span><h3>{{e.title}}</h3><p class="muted">{{e.organizer}} · {{e.event_date}} · {{e.venue}}</p><p>{{e.description}}</p><div class="actions"><form method="post" action="{{url_for('review_event',event_id=e.id)}}" style="display:flex;gap:8px;flex-wrap:wrap"><button name="decision" value="Approved" class="green">Approve</button><button name="decision" value="Rejected" class="red">Reject</button></form></div></div>{% else %}<div class="card">No pending proposals.</div>{% endfor %}</div>"""
        return page("Faculty Dashboard",body,pending=pending,approved=approved)
    # Admin
    stats={ "users":con.execute("SELECT COUNT(*) n FROM users").fetchone()["n"],
            "events":con.execute("SELECT COUNT(*) n FROM events").fetchone()["n"],
            "pending":con.execute("SELECT COUNT(*) n FROM events WHERE status='Pending'").fetchone()["n"],
            "registrations":con.execute("SELECT COUNT(*) n FROM registrations").fetchone()["n"],
            "attendance":con.execute("SELECT COUNT(*) n FROM attendance").fetchone()["n"] }
    events=con.execute("SELECT e.*,u.name organizer FROM events e JOIN users u ON u.id=e.organizer_id ORDER BY e.created_at DESC LIMIT 12").fetchall()
    con.close()
    body="""<section class="hero"><h1>Admin control centre</h1><p>College-wide overview of users, events, registrations and verified attendance.</p><a class="btn secondary" href="{{url_for('admin_users')}}">Manage users</a></section><div class="grid">{% for label,val in stats.items() %}<div class="card"><div class="stat">{{val}}</div><div class="muted">{{label|capitalize}}</div></div>{% endfor %}</div><h2 class="section-title">Recent events</h2><div class="card"><table><tr><th>Event</th><th>Organizer</th><th>Date</th><th>Status</th><th>Analytics</th></tr>{% for e in events %}<tr><td>{{e.title}}</td><td>{{e.organizer}}</td><td>{{e.event_date}}</td><td>{{e.status}}</td><td><a href="{{url_for('event_report',event_id=e.id)}}">View report</a></td></tr>{% endfor %}</table></div>"""
    con.close(); return page("Admin Dashboard",body,stats=stats,events=events)

@APP.route("/events/create", methods=["GET","POST"])
@login_required
@roles_required("Organizer","Admin")
def create_event():
    u=current_user()
    if request.method=="POST":
        fields={k:request.form.get(k,"").strip() for k in ["title","description","category","event_date","start_time","end_time","venue","deadline","eligibility"]}
        try: capacity=int(request.form.get("capacity","0"))
        except: capacity=0
        if not all(fields.values()) or capacity<1 or fields["start_time"]>=fields["end_time"]:
            flash("Complete every field, use capacity above zero, and ensure end time is after start time.")
        else:
            con=db(); warnings=conflict_messages(con,fields["event_date"],fields["start_time"],fields["end_time"],fields["venue"])
            con.execute("""INSERT INTO events(title,description,category,event_date,start_time,end_time,venue,capacity,deadline,eligibility,organizer_id,status,created_at)
             VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)""",(fields["title"],fields["description"],fields["category"],fields["event_date"],fields["start_time"],fields["end_time"],fields["venue"],capacity,fields["deadline"],fields["eligibility"],u["id"],"Pending",datetime.now().isoformat(timespec="seconds")))
            con.commit(); con.close()
            if warnings: flash("Proposal submitted. Conflict warning: "+" ".join(warnings))
            else: flash("Proposal submitted for faculty approval.")
            return redirect(url_for("dashboard"))
    body="""<div class="card"><h2>Create event proposal</h2><p class="muted">New proposals remain pending until faculty approval. Time and venue conflicts are checked against existing pending/approved events.</p><form method="post"><label>Event title<input required name="title"></label><label>Description<textarea required name="description"></textarea></label><div class="row"><label>Category<select name="category">{% for c in ['Workshop','Hackathon','Seminar','Competition','Cultural','Club Event','Career','Other'] %}<option>{{c}}</option>{% endfor %}</select></label><label>Capacity<input required type="number" min="1" name="capacity" value="50"></label></div><div class="row"><label>Date<input required type="date" name="event_date"></label><label>Registration deadline<input required type="date" name="deadline"></label></div><div class="row"><label>Start time<input required type="time" name="start_time"></label><label>End time<input required type="time" name="end_time"></label></div><label>Venue<input required name="venue" placeholder="Seminar Hall A"></label><label>Eligibility<input required name="eligibility" value="All students"></label><button>Submit proposal</button></form></div>"""
    return page("Create Event",body)

@APP.route("/approvals")
@login_required
@roles_required("Faculty","Admin")
def approvals():
    con=db(); events=con.execute("SELECT e.*,u.name organizer FROM events e JOIN users u ON u.id=e.organizer_id WHERE e.status='Pending' ORDER BY e.created_at").fetchall(); con.close()
    body="""<h1>Event approvals</h1><div class="grid">{% for e in events %}<div class="card"><h3>{{e.title}}</h3><p class="muted">{{e.organizer}} · {{e.event_date}} · {{e.start_time}}–{{e.end_time}} · {{e.venue}}</p><p>{{e.description}}</p><form method="post" action="{{url_for('review_event',event_id=e.id)}}" class="actions"><button name="decision" value="Approved">Approve</button><button class="red" name="decision" value="Rejected">Reject</button></form></div>{% else %}<div class="card">No proposals need review.</div>{% endfor %}</div>"""
    return page("Approvals",body,events=events)

@APP.route("/events/<int:event_id>/review",methods=["POST"])
@login_required
@roles_required("Faculty","Admin")
def review_event(event_id):
    decision=request.form.get("decision")
    if decision not in ("Approved","Rejected"): abort(400)
    con=db(); e=con.execute("SELECT * FROM events WHERE id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    if decision=="Approved":
        warnings=conflict_messages(con,e["event_date"],e["start_time"],e["end_time"],e["venue"],event_id)
        if warnings: flash("Approval warning: "+" ".join(warnings))
    con.execute("UPDATE events SET status=? WHERE id=?",(decision,event_id)); con.commit(); con.close()
    flash("Event "+decision.lower()+"."); return redirect(url_for("approvals"))

@APP.route("/events/<int:event_id>")
def event_detail(event_id):
    con=db(); e=con.execute("SELECT e.*,u.name organizer FROM events e JOIN users u ON u.id=e.organizer_id WHERE e.id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    u=current_user()
    reg=None; count=con.execute("SELECT COUNT(*) n FROM registrations WHERE event_id=?",(event_id,)).fetchone()["n"]
    if u and u["role"]=="Student": reg=con.execute("SELECT * FROM registrations WHERE event_id=? AND student_id=?",(event_id,u["id"])).fetchone()
    avg=con.execute("SELECT AVG(rating) a,COUNT(*) n FROM feedback WHERE event_id=?",(event_id,)).fetchone()
    con.close()
    body="""<div class="card"><span class="pill">{{e.category}} · {{e.status}}</span><h1>{{e.title}}</h1><p>{{e.description}}</p><div class="grid"><div><b>Date</b><p>{{e.event_date}}</p></div><div><b>Time</b><p>{{e.start_time}}–{{e.end_time}}</p></div><div><b>Venue</b><p>{{e.venue}}</p></div><div><b>Capacity</b><p>{{count}} / {{e.capacity}} registered</p></div></div><p class="muted">Organizer: {{e.organizer}} · Deadline: {{e.deadline}} · Eligibility: {{e.eligibility}}</p><p class="muted">Feedback average: {{'%.1f'|format(avg.a) if avg.a else 'Not rated yet'}} / 5 ({{avg.n}} ratings)</p><div class="actions">{% if user and user.role=='Student' and e.status=='Approved' %}{% if reg %}<a class="btn green" href="{{url_for('qr_pass',registration_id=reg.id)}}">My QR pass</a>{% else %}<form method="post" action="{{url_for('register_event',event_id=e.id)}}"><button>Register for event</button></form>{% endif %}{% elif not user %}<a class="btn" href="{{url_for('login')}}">Login to register</a>{% endif %}{% if user and user.role in ['Organizer','Admin'] %}<a class="btn secondary" href="{{url_for('event_report',event_id=e.id)}}">Event analytics</a>{% endif %}</div></div>"""
    return page(e["title"],body,e=e,count=count,reg=reg,avg=avg)

@APP.route("/events/<int:event_id>/register",methods=["POST"])
@login_required
@roles_required("Student")
def register_event(event_id):
    u=current_user(); con=db(); e=con.execute("SELECT * FROM events WHERE id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    if e["status"]!="Approved": flash("Only approved events can be registered.")
    elif e["deadline"] < datetime.now().date().isoformat(): flash("Registration deadline has passed.")
    elif con.execute("SELECT COUNT(*) n FROM registrations WHERE event_id=?",(event_id,)).fetchone()["n"]>=e["capacity"]: flash("This event has reached capacity.")
    else:
        token=secrets.token_urlsafe(24)
        try:
            con.execute("INSERT INTO registrations(event_id,student_id,qr_token,registered_at) VALUES(?,?,?,?)",(event_id,u["id"],token,datetime.now().isoformat(timespec="seconds")))
            con.commit(); flash("Registration successful. Your QR pass is ready.")
        except sqlite3.IntegrityError: flash("You are already registered for this event.")
    con.close(); return redirect(url_for("event_detail",event_id=event_id))

@APP.route("/registrations/<int:registration_id>/qr")
@login_required
def qr_pass(registration_id):
    u=current_user(); con=db()
    r=con.execute("""SELECT r.*,e.title,e.event_date,e.venue,r.student_id FROM registrations r JOIN events e ON e.id=r.event_id WHERE r.id=?""",(registration_id,)).fetchone()
    if not r: con.close(); abort(404)
    if u["role"] not in ("Admin","Organizer","Faculty") and r["student_id"]!=u["id"]: con.close(); abort(403)
    con.close()
    img=qrcode.make(r["qr_token"]); bio=io.BytesIO(); img.save(bio,format="PNG"); bio.seek(0)
    return send_file(bio,mimetype="image/png",as_attachment=True,download_name=f"Event2Career_QR_{registration_id}.png")

@APP.route("/attendance",methods=["GET","POST"])
@login_required
@roles_required("Organizer","Faculty","Admin")
def attendance_scan():
    result=None
    if request.method=="POST":
        token=request.form.get("token","").strip()
        con=db(); r=con.execute("""SELECT r.*,e.title,e.organizer_id FROM registrations r JOIN events e ON e.id=r.event_id WHERE r.qr_token=?""",(token,)).fetchone()
        if not r: result=("error","Invalid QR token.")
        elif current_user()["role"]=="Organizer" and r["organizer_id"]!=current_user()["id"]: result=("error","You can only verify attendance for your own events.")
        elif con.execute("SELECT id FROM attendance WHERE registration_id=?",(r["id"],)).fetchone(): result=("warning","Attendance was already recorded for this pass.")
        else:
            con.execute("INSERT INTO attendance(registration_id,scanned_at) VALUES(?,?)",(r["id"],datetime.now().isoformat(timespec="seconds")))
            con.commit(); result=("success",f"Attendance verified for event: {r['title']}.")
        con.close()
    body="""<div class="card" style="max-width:700px"><h1>QR attendance verification</h1><p class="muted">Scan the attendee's QR with a QR reader and paste the token here. This demo verifies the unique token; only authorized staff should use this page.</p><form method="post"><label>QR token<input name="token" required placeholder="Paste token from QR reader"></label><button>Verify and mark attendance</button></form>{% if result %}<div class="alert">{{result[1]}}</div>{% endif %}</div>"""
    return page("Attendance",body,result=result)

@APP.route("/feedback/<int:event_id>",methods=["POST"])
@login_required
@roles_required("Student")
def feedback(event_id):
    u=current_user()
    try: rating=int(request.form.get("rating","5"))
    except: rating=0
    comments=request.form.get("comments","").strip()
    con=db()
    registered=con.execute("SELECT id FROM registrations WHERE event_id=? AND student_id=?",(event_id,u["id"])).fetchone()
    if not registered: flash("Register for the event before submitting feedback.")
    elif not 1<=rating<=5: flash("Rating must be between 1 and 5.")
    else:
        try:
            con.execute("INSERT INTO feedback(event_id,student_id,rating,comments,submitted_at) VALUES(?,?,?,?,?)",(event_id,u["id"],rating,comments,datetime.now().isoformat(timespec="seconds")))
            con.commit(); flash("Thank you. Feedback saved.")
        except sqlite3.IntegrityError: flash("You have already submitted feedback for this event.")
    con.close(); return redirect(url_for("dashboard"))

@APP.route("/events/<int:event_id>/report")
@login_required
@roles_required("Organizer","Faculty","Admin")
def event_report(event_id):
    u=current_user(); con=db(); e=con.execute("SELECT * FROM events WHERE id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    if u["role"]=="Organizer" and e["organizer_id"]!=u["id"]: con.close(); abort(403)
    total=con.execute("SELECT COUNT(*) n FROM registrations WHERE event_id=?",(event_id,)).fetchone()["n"]
    attended=con.execute("""SELECT COUNT(*) n FROM attendance a JOIN registrations r ON r.id=a.registration_id WHERE r.event_id=?""",(event_id,)).fetchone()["n"]
    avg=con.execute("SELECT AVG(rating) a,COUNT(*) n FROM feedback WHERE event_id=?",(event_id,)).fetchone()
    people=con.execute("""SELECT u.name,u.email,r.registered_at,a.scanned_at,c.certificate_no,r.id rid
      FROM registrations r JOIN users u ON u.id=r.student_id LEFT JOIN attendance a ON a.registration_id=r.id
      LEFT JOIN certificates c ON c.registration_id=r.id WHERE r.event_id=? ORDER BY u.name""",(event_id,)).fetchall()
    con.close()
    body="""<section class="hero"><h1>{{e.title}} — analytics</h1><p>Event-level participation and feedback overview.</p></section><div class="grid"><div class="card"><div class="stat">{{total}}</div><div class="muted">Registrations</div></div><div class="card"><div class="stat">{{attended}}</div><div class="muted">Verified attendees</div></div><div class="card"><div class="stat">{{'%.1f'|format(attended*100/total) if total else '0.0'}}%</div><div class="muted">Attendance rate</div></div><div class="card"><div class="stat">{{'%.1f'|format(avg.a) if avg.a else '—'}} / 5</div><div class="muted">Average feedback ({{avg.n}} ratings)</div></div></div><h2 class="section-title">Participants</h2><div class="card"><div class="actions"><form method="post" action="{{url_for('issue_certificates',event_id=e.id)}}"><button>Issue certificates to verified attendees</button></form><a class="btn secondary" href="{{url_for('export_event',event_id=e.id)}}">Export CSV</a><a class="btn secondary" href="{{url_for('attendance_scan')}}">QR attendance</a></div><table><tr><th>Name</th><th>Email</th><th>Registered</th><th>Attendance</th><th>Certificate</th><th>QR</th></tr>{% for p in people %}<tr><td>{{p.name}}</td><td>{{p.email}}</td><td>{{p.registered_at[:10]}}</td><td>{{'Yes' if p.scanned_at else 'No'}}</td><td>{{p.certificate_no or 'Not issued'}}</td><td><a href="{{url_for('qr_pass',registration_id=p.rid)}}">QR</a></td></tr>{% endfor %}</table></div>"""
    return page("Event Analytics",body,e=e,total=total,attended=attended,avg=avg,people=people)

@APP.route("/events/<int:event_id>/certificates",methods=["POST"])
@login_required
@roles_required("Organizer","Admin","Faculty")
def issue_certificates(event_id):
    u=current_user(); con=db(); e=con.execute("SELECT * FROM events WHERE id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    if u["role"]=="Organizer" and e["organizer_id"]!=u["id"]: con.close(); abort(403)
    regs=con.execute("SELECT r.id FROM registrations r JOIN attendance a ON a.registration_id=r.id WHERE r.event_id=?",(event_id,)).fetchall()
    issued=0
    for r in regs:
        if not con.execute("SELECT id FROM certificates WHERE registration_id=?",(r["id"],)).fetchone():
            con.execute("INSERT INTO certificates(registration_id,certificate_no,issued_at) VALUES(?,?,?)",(r["id"],"E2C-"+secrets.token_hex(5).upper(),datetime.now().isoformat(timespec="seconds"))); issued+=1
    con.commit(); con.close(); flash(f"Issued {issued} new certificate(s) to verified attendees."); return redirect(url_for("event_report",event_id=event_id))

@APP.route("/registrations/<int:registration_id>/certificate")
@login_required
def certificate(registration_id):
    u=current_user(); con=db()
    row=con.execute("""SELECT c.certificate_no,c.issued_at,e.title,e.event_date,u.name,r.student_id
      FROM certificates c JOIN registrations r ON r.id=c.registration_id JOIN events e ON e.id=r.event_id
      JOIN users u ON u.id=r.student_id WHERE r.id=?""",(registration_id,)).fetchone()
    if not row: con.close(); abort(404)
    if current_user()["role"]!="Admin" and row["student_id"]!=current_user()["id"]: con.close(); abort(403)
    con.close()
    content=f"""EVENT2CAREER — CERTIFICATE OF PARTICIPATION

This is to certify that {row['name']} participated in
{row['title']} held on {row['event_date']}.

Certificate Number: {row['certificate_no']}
Issued on: {row['issued_at'][:10]}

Event2Career College Event Management System
Academic prototype certificate — verify with the issuing institution.
"""
    return send_file(io.BytesIO(content.encode()),mimetype="text/plain",as_attachment=True,download_name=f"{row['certificate_no']}.txt")

@APP.route("/events/<int:event_id>/export.csv")
@login_required
@roles_required("Organizer","Faculty","Admin")
def export_event(event_id):
    u=current_user(); con=db(); e=con.execute("SELECT * FROM events WHERE id=?",(event_id,)).fetchone()
    if not e: con.close(); abort(404)
    if u["role"]=="Organizer" and e["organizer_id"]!=u["id"]: con.close(); abort(403)
    rows=con.execute("""SELECT u.name,u.email,r.registered_at,a.scanned_at,c.certificate_no
      FROM registrations r JOIN users u ON u.id=r.student_id LEFT JOIN attendance a ON a.registration_id=r.id
      LEFT JOIN certificates c ON c.registration_id=r.id WHERE r.event_id=?""",(event_id,)).fetchall(); con.close()
    out=io.StringIO(); w=csv.writer(out); w.writerow(["Name","Email","Registered At","Attended At","Certificate Number"])
    for r in rows: w.writerow(list(r))
    return send_file(io.BytesIO(out.getvalue().encode("utf-8-sig")),mimetype="text/csv",as_attachment=True,download_name=f"event_{event_id}_participants.csv")

@APP.route("/admin/users")
@login_required
@roles_required("Admin")
def admin_users():
    con=db(); users=con.execute("SELECT id,name,email,role,department,interests FROM users ORDER BY role,name").fetchall(); con.close()
    body="""<h1>User management</h1><p class="muted">Change role assignments for this demonstration. In a production deployment, role changes should be audited and restricted.</p><div class="card"><table><tr><th>Name</th><th>Email</th><th>Department</th><th>Role</th><th>Update role</th></tr>{% for x in users %}<tr><td>{{x.name}}</td><td>{{x.email}}</td><td>{{x.department}}</td><td>{{x.role}}</td><td><form method="post" action="{{url_for('set_role',user_id=x.id)}}" style="display:flex;gap:6px"><select name="role">{% for r in roles %}<option {% if x.role==r %}selected{% endif %}>{{r}}</option>{% endfor %}</select><button>Save</button></form></td></tr>{% endfor %}</table></div>"""
    return page("User Management",body,users=users,roles=ROLES)

@APP.route("/admin/users/<int:user_id>/role",methods=["POST"])
@login_required
@roles_required("Admin")
def set_role(user_id):
    role=request.form.get("role")
    if role not in ROLES: abort(400)
    if user_id==current_user()["id"] and role!="Admin": flash("You cannot remove your own Admin role.")
    else:
        con=db(); con.execute("UPDATE users SET role=? WHERE id=?",(role,user_id)); con.commit(); con.close(); flash("User role updated.")
    return redirect(url_for("admin_users"))

@APP.errorhandler(403)
def forbidden(e): return page("Access denied","<div class='card'><h1>403 — Access denied</h1><p>Your account does not have permission to open this page.</p><a class='btn' href='/'>Return home</a></div>"),403
@APP.errorhandler(404)
def missing(e): return page("Not found","<div class='card'><h1>404 — Not found</h1><p>The requested item could not be found.</p><a class='btn' href='/'>Return home</a></div>"),404

init_db()
if __name__ == "__main__":
    APP.run(host="0.0.0.0", port=5000, debug=False)


Writing app.py


In [3]:
import subprocess, time, os, signal
# Stop an earlier server if this cell is rerun
subprocess.run("pkill -f 'python app.py' || true", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
server_log = open("event2career_server.log", "w")
server = subprocess.Popen(["python", "app.py"], stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(3)
print("Flask process started. Local app: http://127.0.0.1:5000")
print("If it fails to start, inspect event2career_server.log.")

Flask process started. Local app: http://127.0.0.1:5000
If it fails to start, inspect event2career_server.log.


In [4]:
from pyngrok import ngrok
from getpass import getpass
token = getpass("Paste your ngrok authtoken (input is hidden): ")
ngrok.set_auth_token(token)
public_url = ngrok.connect(5000, "http").public_url
print("Open Event2Career here:", public_url)

Paste your ngrok authtoken (input is hidden): ··········
Open Event2Career here: https://truce-backup-grew.ngrok-free.dev
